# Practice: a weekend trip planner agent

You and a friend live in Berlin and want a cheap trip this weekend: Prague or Vienna, 300 EUR for the two of you. In this workbook you build an agent that plans it, **one step at a time**. You ask the same question after every step and watch the answer get better.

| Step | What we add | What the agent can do afterwards |
|---|---|---|
| 1 | An agent with no tools | Answer from what the model learned in training, and guess the rest |
| 2 | A first tool | Know the dates of this weekend |
| 3 | A tool that calls a web service | Know the real weather forecast |
| 4 | A knowledge base | Look up fares and hostel prices in our travel guide |
| 5 | A calculator and rules | Check the budget exactly and follow our rules |
| 6 | Memory | Answer a follow-up question |
| 7 | A team of agents (Part 2) | Research both cities at the same time |

After this workbook you can explain what a tool is and who runs it, read an agent's steps, give an agent knowledge and memory, and say when a team of agents is worth it.

> **Note.** The saved outputs come from a run on 3 November 2026 with Claude Haiku 5.5. Claude's answers change from run to run, and so does the weather.

## Setup: the key and the model

The agent uses Claude through the Claude Platform API. An **API key** is a secret that identifies you and is billed for every request. It never goes into a notebook. It goes into a file `.env` in the `workbooks` folder, with one line `ANTHROPIC_API_KEY=your key`. Copy `.env.example` to `.env` and paste your key; git ignores `.env`.

- `load_dotenv(find_dotenv(usecwd=True))` from `python-dotenv` finds the file, starting from the folder where the notebook runs, and loads the key into the environment.
- If no key is found, `getpass` asks for it without showing it on screen.

In [1]:
import os
from getpass import getpass

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your Anthropic API key: ")
print("API key found. Its value is not printed.")

API key found. Its value is not printed.


`ChatAnthropic` from `langchain-anthropic` is the model object. We use **Claude Haiku 5.5** (`claude-haiku-5-5`), the smallest and cheapest Claude model, because an agent calls the model many times for one question. `max_tokens` is the longest reply allowed; always set it.

In [2]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(model="claude-haiku-5-5", max_tokens=4096)

## A model call or an agent?

We already have `model`, and `model.invoke(...)` sends a question to Claude. So why build an agent? Both send our question to the same model. The difference is **what happens after Claude replies**.

`create_agent(model=model, tools=tools, system_prompt=...)` from `langchain.agents` builds an agent around the model. A **system prompt** is the developer's instruction for the whole conversation; a **user message** is what the user asks. We name both in full: `SYSTEM_PROMPT` and `USER_MESSAGE_GOAL`.

| | `model.invoke(messages)` | `agent.invoke({"messages": messages})` |
|---|---|---|
| Input | a text, or a list of `(role, text)` messages | a dictionary with the key `"messages"` |
| Output | one message (`AIMessage`); `.text` is the answer | a dictionary; `result["messages"]` is the whole conversation, and its last message is the answer |
| Model calls | exactly one | one or more, until the model stops asking for tools |
| Tools | the model can only *ask* for a tool; our own code must run it and call the model again | the agent runs the tool and calls the model again by itself |
| System prompt | sent as a `("system", ...)` message in every call | given once, to `create_agent` |

**A model call** is one request and one reply:

```mermaid
graph LR;
    user_message[User message] --> claude[Claude: one model call];
    claude --> reply[Reply: one AIMessage];
```

**An agent** is a loop around the model. After every reply it checks: did the model ask for a tool? If yes, it runs the tool, adds the result to the conversation and calls the model again. If no, the last reply is the answer.

```mermaid
graph LR;
    user_message[User message] --> model_node[Model call];
    model_node --> decision{Tool calls in the reply?};
    decision -- yes --> tools_node[Run the tools in our Python];
    tools_node -- results added to the messages --> model_node;
    decision -- no --> answer[Answer: the last message];
```

The cell runs the same question both ways, with no tools, and prints what each call returns.

In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = "You plan short, cheap trips for students who live in Berlin."
USER_MESSAGE_GOAL = "Plan a cheap trip from Berlin for this weekend: 2 people, 300 EUR in total. Prague or Vienna?"

# 1. The model alone: one call, one reply.
reply = model.invoke([("system", SYSTEM_PROMPT), ("user", USER_MESSAGE_GOAL)])
print("model.invoke returns:", type(reply).__name__)

# 2. The same model inside an agent, still with no tools.
agent = create_agent(model=model, tools=[], system_prompt=SYSTEM_PROMPT)
result = agent.invoke({"messages": [("user", USER_MESSAGE_GOAL)]})
print("agent.invoke returns:", type(result).__name__, "with the keys", list(result))
print("messages in result  :", [message.type for message in result["messages"]])

`message.type` is `"human"` for a user message and `"ai"` for a model reply. With no tools, the agent made one model call, just like `model.invoke`: the conversation is the question and one reply. So without tools, an agent is only a model call in a different wrapper. The difference starts in Step 2, when the model can ask for a tool: `model.invoke` stops at the request, and the agent goes on.

## Step 1: an agent with no tools

The agent from the cell above has no tools, so it is only the model. Its last message is the answer; `.text` gives the text of a message.

In [3]:
print(result["messages"][-1].text)

**Prague.** It fits your budget much more comfortably, and it's a shorter trip.

**Getting there:** Berlin to Prague is about 4-5 hours by train or FlixBus. Vienna is about 8-9 hours, which eats most of a weekend.

**Rough budget for 2 people, 2 nights (estimates, check current prices):**

| Item | Estimate (total for 2) |
|---|---|
| Return transport (FlixBus or Deutsche Bahn Sparpreis, booked early) | 60-100 EUR |
| Hostel dorms or a budget double room, 2 nights | 80-120 EUR |
| Food and drinks (Czech lunch menus and beer are cheap) | 60-80 EUR |
| Public transport and small extras | 15-20 EUR |
| **Total** | **215-320 EUR** |

Vienna would likely run 350 EUR or more for the same trip, so it's tight on your budget.

**Tips to keep it under 300 EUR:**
- Book transport 2-4 weeks ahead if you can. Prices rise sharply close to the date.
- Take the FlixBus if the train fares are high. Both are usually cheap.
- Stay in a hostel or a private room outside the Old Town. It's cheaper, and a 10

**Discuss.** The answer may sound good. Now check it line by line. Which of these could the model really know?

- the dates of "this weekend"
- the weather on those days
- bus fares and hostel prices today
- whether the plan fits 300 EUR

None of them: a model knows only its training data, and it does not know today's date. It either says so or **guesses**. Each of the next steps fixes one of these gaps with a tool.

## Step 2: a first tool

### What a tool is

A **tool** is a Python function that the model may ask to run. The model never runs it. It writes a request with the tool's name and arguments; LangChain runs the function in *our* Python; the result goes back to the model as text.

The `@tool` decorator from `langchain.tools` turns a function into a tool. Three parts of it reach the model: the **name** (the function name), the **description** (the docstring) and the **arguments** (from the type hints). So the docstring is an instruction for the model: say when to use the tool.

Our first tool needs no internet: it computes the coming Saturday and Sunday from the computer's clock.

- `date.today()` is today's date; `.weekday()` gives 0 for Monday up to 6 for Sunday.
- `(5 - weekday) % 7` is the number of days until Saturday; `timedelta(days=n)` is a step of n days.
- The tool returns JSON text with `json.dumps`, which a model reads well.

In [4]:
import json
from datetime import date, timedelta

from langchain.tools import tool


@tool
def next_weekend() -> str:
    """Return today's date and the dates of the coming Saturday and Sunday. Call it when the user says "this weekend"."""
    today = date.today()
    saturday = today + timedelta(days=(5 - today.weekday()) % 7)
    sunday = saturday + timedelta(days=1)
    return json.dumps({"today": today.isoformat(), "saturday": saturday.isoformat(), "sunday": sunday.isoformat()})

Test every tool by hand before a model gets it: `.invoke()` with a dictionary of arguments calls it the same way the agent will. A bug in a tool looks like a mistake of the model later.

In [5]:
print(next_weekend.invoke({}))
print(next_weekend.name, "|", next_weekend.description)

{"today": "2026-11-03", "saturday": "2026-11-07", "sunday": "2026-11-08"}
next_weekend | Return today's date and the dates of the coming Saturday and Sunday. Call it when the user says "this weekend".


### Reading the agent's steps

To see what the agent did, we print its conversation. A model message (`type == "ai"`) can hold **tool calls** in `.tool_calls`: each is a dictionary with the tool's `name` and `args`. A tool message (`type == "tool"`) holds the result. The helper `show_run` prints both, then the answer.

In [6]:
def show_run(result: dict) -> None:
    """Print every tool request of the model, every tool result, and the final answer."""
    for message in result["messages"]:
        if message.type == "ai":
            for call in message.tool_calls:
                print(f"model asks for : {call['name']}({call['args']})")
        elif message.type == "tool":
            print(f"tool returns   : {message.content[:150]}")
    print("\nANSWER:\n" + result["messages"][-1].text)

**Predict.** The agent now has the tool `next_weekend`. Will it call it before it answers?

In [7]:
agent = create_agent(model=model, tools=[next_weekend], system_prompt=SYSTEM_PROMPT)
show_run(agent.invoke({"messages": [("user", USER_MESSAGE_GOAL)]}))

model asks for : next_weekend({})
tool returns   : {"today": "2026-11-03", "saturday": "2026-11-07", "sunday": "2026-11-08"}

ANSWER:
**Prague is the better pick** for 300 EUR for two people.

- **Distance and travel:** Prague is about 350 km from Berlin, roughly 4.5 to 5 hours by bus or train. Vienna is about 680 km, so expect 8 to 9 hours. Less travel time means less money spent on transport and more of the weekend in the city.
- **Transport:** Buses (FlixBus and similar) to Prague often cost 15 to 40 EUR each way if you book early, so a return trip for two might be 60 to 120 EUR total. Vienna tends to cost more, and the trip is long enough that you might want to pay for a sleeper or a faster train.
- **Accommodation:** Hostels and budget hotels in Prague are generally cheaper than in Vienna. Two nights in a shared hostel room or a basic double room is realistic within budget.
- **Food and activities:** Prague is cheaper for eating out and for sights, though Vienna's museums and coff

The model chose to call the tool, read the dates, and then answered. We did not tell it to: the docstring said when the tool is useful. This loop is what makes an **agent**: the model reads, asks for a tool, reads the result, and decides again, until it can answer. The dates are now right, but the weather and the prices are still guesses.

Who does what in this run: Claude only writes text, including the tool request; our Python runs the tool.

```mermaid
sequenceDiagram
    participant User
    participant Agent as Agent loop in our Python
    participant Claude
    participant Tool as next_weekend in our Python
    User->>Agent: USER_MESSAGE_GOAL
    Agent->>Claude: model call 1: question and tool descriptions
    Claude-->>Agent: tool request next_weekend
    Agent->>Tool: run next_weekend
    Tool-->>Agent: dates as JSON
    Agent->>Claude: model call 2: question, request and dates
    Claude-->>Agent: answer, no tool request
    Agent-->>User: last message
```

### The model alone, with the same tool

Now the comparison from the start of the workbook, with a tool. `model.bind_tools(tools)` gives the model the tool descriptions, without an agent around it. **Predict.** What does `model.invoke` return now?

In [ ]:
model_with_tools = model.bind_tools([next_weekend])
reply = model_with_tools.invoke([("system", SYSTEM_PROMPT), ("user", USER_MESSAGE_GOAL)])
print("tool calls:", reply.tool_calls)
print("text      :", reply.text or "(empty: no answer yet)")

The model alone stops after model call 1: it returns the tool request, nobody runs the tool, and we get no answer. To finish, our own code would have to run `next_weekend`, add the result to the messages and call the model again, and repeat this for every tool request. This loop is exactly what `create_agent` writes for us.

## Step 3: a tool that calls a web service

The weather changes every day, so it must come from a web service. **Open-Meteo** gives forecasts for free, without a key, for non-commercial use such as teaching. The data is under the CC BY 4.0 licence, so the tool returns the credit "Weather data by Open-Meteo.com".

The tool makes two requests:

1. the **geocoding** service turns the city name into latitude and longitude;
2. the **forecast** service gives, per day, the lowest and highest temperature and the highest chance of rain.

`httpx.get(url, params=..., timeout=10)` sends a request; `params` becomes the query string, and `timeout` stops waiting after 10 seconds. `.json()` reads the answer into a dictionary. `zip(a, b, c)` walks through several lists side by side.

If something goes wrong, the tool returns a sentence that starts with "Error", instead of crashing. The model reads this sentence like any other result and can react to it.

In [8]:
import httpx

GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"


@tool(parse_docstring=True)
def weather_forecast(city: str, start_date: str, end_date: str) -> str:
    """Get the daily weather forecast for a city. Works only for dates up to 16 days from today.

    Args:
        city: City name in English, for example "Prague" (not "Praha").
        start_date: First day, as YYYY-MM-DD.
        end_date: Last day, as YYYY-MM-DD.
    """
    try:
        places = httpx.get(GEOCODING_URL, params={"name": city, "count": 1, "language": "en"}, timeout=10).json()
        if not places.get("results"):
            return f"Error: no city called {city!r} was found. Try the English name."
        place = places["results"][0]
        response = httpx.get(FORECAST_URL, timeout=10, params={
            "latitude": place["latitude"], "longitude": place["longitude"], "timezone": "auto",
            "start_date": start_date, "end_date": end_date,
            "daily": "temperature_2m_min,temperature_2m_max,precipitation_probability_max",
        })
        response.raise_for_status()
    except httpx.HTTPError as error:
        return f"Error: the weather service did not answer ({type(error).__name__}). It forecasts at most 16 days ahead."
    daily = response.json()["daily"]
    days = [
        {"date": day, "min_c": low, "max_c": high, "rain_chance_pct": rain}
        for day, low, high, rain in zip(daily["time"], daily["temperature_2m_min"],
                                        daily["temperature_2m_max"], daily["precipitation_probability_max"])
    ]
    return json.dumps({"city": place["name"], "country": place["country"], "days": days,
                       "source": "Weather data by Open-Meteo.com"})

`parse_docstring=True` reads the `Args:` section, so each argument gets its own description for the model. `raise_for_status()` turns an answer with an error status into an exception, which the `except` catches. Test the tool by hand for this weekend:

In [9]:
weekend = json.loads(next_weekend.invoke({}))
print(weather_forecast.invoke({"city": "Prague", "start_date": weekend["saturday"], "end_date": weekend["sunday"]}))

{"city": "Prague", "country": "Czechia", "days": [{"date": "2026-11-07", "min_c": 1.5, "max_c": 6.6, "rain_chance_pct": 6}, {"date": "2026-11-08", "min_c": -0.8, "max_c": 4.6, "rain_chance_pct": 9}], "source": "Weather data by Open-Meteo.com"}


**Predict.** The weather tool needs dates, but the user did not give any. What will the agent do first?

In [10]:
agent = create_agent(model=model, tools=[next_weekend, weather_forecast], system_prompt=SYSTEM_PROMPT)
show_run(agent.invoke({"messages": [("user", USER_MESSAGE_GOAL)]}))

model asks for : next_weekend({})
tool returns   : {"today": "2026-11-03", "saturday": "2026-11-07", "sunday": "2026-11-08"}
model asks for : weather_forecast({'city': 'Prague', 'start_date': '2026-11-07', 'end_date': '2026-11-08'})
model asks for : weather_forecast({'city': 'Vienna', 'start_date': '2026-11-07', 'end_date': '2026-11-08'})
tool returns   : {"city": "Prague", "country": "Czechia", "days": [{"date": "2026-11-07", "min_c": 1.5, "max_c": 6.6, "rain_chance_pct": 6}, {"date": "2026-11-08", "mi
tool returns   : {"city": "Vienna", "country": "Austria", "days": [{"date": "2026-11-07", "min_c": 1.6, "max_c": 9.0, "rain_chance_pct": 5}, {"date": "2026-11-08", "mi

ANSWER:
**Recommendation: Prague**, mainly because it should be cheaper. I can't look up live fares or hostel prices, so the cost figures below are rough estimates to verify before booking.

**Weather, Sat 7 – Sun 8 Nov:**
- **Prague:** 1.5–6.6 °C on Saturday, -0.8–4.6 °C on Sunday. Rain chance 6% and 9%.
- **Vienna:** 1

Read the `model asks for` lines. The model first asked for the dates, then passed them to the weather tool: the result of one tool became the input of the next. It may also ask for both cities in one step (**parallel tool calls**), because the two forecasts do not depend on each other. The weather is now real. The fares and hostel prices are still guesses.

## Step 4: a knowledge base

### Our own travel guide

Fares and hostel prices are not in any free web service. Many real agents have the same problem: the facts they need are in the company's own documents. The answer is a **knowledge base**: our own texts, and a tool that searches them. This is the retrieval step of RAG from notebook 01 of this session, used as a tool: the model decides *when* to look something up.

Our knowledge base is a short travel guide, one note per city. **The guide is written for this course: the prices are rough example values, not real offers.** Check real prices before you book.

In [11]:
TRIP_GUIDE = [
    "Prague, Czechia. Currency: Czech koruna (CZK). Bus from Berlin: about 4.5 hours, return fare about 40 EUR per person. "
    "Hostel bed: about 25 EUR per night. Cheap food: a lunch menu costs about 8 EUR. "
    "Free: Charles Bridge early in the morning, and the view from Letna Park.",
    "Vienna, Austria. Currency: euro (EUR). Bus from Berlin: about 9.5 hours, return fare about 70 EUR per person. "
    "Hostel bed: about 35 EUR per night. Cheap food: a sausage stand costs about 5 EUR. "
    "Free: the flea market at the Naschmarkt on Saturday morning.",
    "Dresden, Germany. Currency: euro (EUR). Bus from Berlin: about 2.5 hours, return fare about 25 EUR per person. "
    "Hostel bed: about 25 EUR per night. Free: a walk along the Elbe meadows, and the Neustadt district.",
]

### A simple search tool

Notebook 01 searches with embeddings. For three notes, a **keyword search** is enough and easy to follow:

- `words(text)` turns a text into a set of lower-case words; `re.findall(r"[a-z]+", ...)` finds all runs of letters.
- `query_words & words(note)` is the set of words that the query and the note share. The note with the most shared words ranks first.
- `sorted(..., key=..., reverse=True)` sorts from the highest score down. The tool returns the best two notes that share at least one word.

In [12]:
import re


def words(text: str) -> set[str]:
    """The set of lower-case words in a text."""
    return set(re.findall(r"[a-z]+", text.lower()))


@tool
def search_trip_guide(query: str) -> str:
    """Search our student travel guide for bus fares, travel times, hostel prices and free things to do. Put the city name in the query."""
    query_words = words(query)
    ranked = sorted(TRIP_GUIDE, key=lambda note: len(query_words & words(note)), reverse=True)
    best_notes = [note for note in ranked[:2] if query_words & words(note)]
    if not best_notes:
        return "No note matches. The guide covers Prague, Vienna and Dresden."
    return "\n\n".join(best_notes)


print(search_trip_guide.invoke({"query": "hostel price Prague"}))

Prague, Czechia. Currency: Czech koruna (CZK). Bus from Berlin: about 4.5 hours, return fare about 40 EUR per person. Hostel bed: about 25 EUR per night. Cheap food: a lunch menu costs about 8 EUR. Free: Charles Bridge early in the morning, and the view from Letna Park.

Vienna, Austria. Currency: euro (EUR). Bus from Berlin: about 9.5 hours, return fare about 70 EUR per person. Hostel bed: about 35 EUR per night. Cheap food: a sausage stand costs about 5 EUR. Free: the flea market at the Naschmarkt on Saturday morning.


**Predict.** The model now has three tools. For the same question, which tools will it call, and how often?

In [13]:
agent = create_agent(model=model, tools=[next_weekend, weather_forecast, search_trip_guide], system_prompt=SYSTEM_PROMPT)
show_run(agent.invoke({"messages": [("user", USER_MESSAGE_GOAL)]}))

model asks for : next_weekend({})
model asks for : search_trip_guide({'query': 'Prague bus fares from Berlin, travel time, hostel prices, free things to do'})
model asks for : search_trip_guide({'query': 'Vienna bus fares from Berlin, travel time, hostel prices, free things to do'})
tool returns   : {"today": "2026-11-03", "saturday": "2026-11-07", "sunday": "2026-11-08"}
tool returns   : Prague, Czechia. Currency: Czech koruna (CZK). Bus from Berlin: about 4.5 hours, return fare about 40 EUR per person. Hostel bed: about 25 EUR per nig
tool returns   : Vienna, Austria. Currency: euro (EUR). Bus from Berlin: about 9.5 hours, return fare about 70 EUR per person. Hostel bed: about 35 EUR per night. Chea
model asks for : weather_forecast({'city': 'Prague', 'start_date': '2026-11-07', 'end_date': '2026-11-08'})
model asks for : weather_forecast({'city': 'Vienna', 'start_date': '2026-11-07', 'end_date': '2026-11-08'})
tool returns   : {"city": "Prague", "country": "Czechia", "days": [{"date

The prices in the answer now come from our guide, not from the model's memory. Check one number from the answer against the `tool returns` lines. One gap is left: the agent adds up fares and beds in its head, and nothing stops it from recommending a plan over the budget.

## Step 5: a calculator and rules

### A budget tool

A language model can make arithmetic mistakes, and it can give a different sum each time. A sum that must be right belongs in Python. The tool `budget_per_person` takes the fare and the bed price from the guide and returns what is left per person for food and activities, and whether the plan fits.

In [14]:
@tool(parse_docstring=True)
def budget_per_person(total_budget_eur: float, people: int, nights: int,
                      return_fare_eur: float, bed_per_night_eur: float) -> str:
    """Compute what is left per person for food and activities after the fare and the beds.

    Args:
        total_budget_eur: Budget of the whole group, in euros.
        people: Number of travellers.
        nights: Number of nights away.
        return_fare_eur: Return fare for one person, in euros.
        bed_per_night_eur: Price of one bed for one night, in euros.
    """
    per_person = total_budget_eur / people
    left = per_person - return_fare_eur - nights * bed_per_night_eur
    return json.dumps({"budget_per_person_eur": round(per_person, 2),
                       "left_for_food_and_activities_eur": round(left, 2), "within_budget": left >= 0})


print(budget_per_person.invoke({"total_budget_eur": 300, "people": 2, "nights": 1,
                                "return_fare_eur": 40, "bed_per_night_eur": 25}))

{"budget_per_person_eur": 150.0, "left_for_food_and_activities_eur": 85.0, "within_budget": true}


150 EUR per person, minus 40 EUR for the bus and 25 EUR for one night, leaves 85 EUR for Prague.

### Rules in the system prompt

The system prompt is where the **rules** for the agent go. Good rules are concrete and can be checked: "be careful with money" is not; "never recommend a plan over the budget" is.

In [15]:
TOOLS = [next_weekend, weather_forecast, search_trip_guide, budget_per_person]
RULES = """You plan short, cheap trips for students who live in Berlin.
Rules:
- Use the tools for every fact: dates, weather, fares, prices and sums. Never guess a number.
- Prices from the travel guide are course examples. Say so in your answer.
- Never recommend a plan over the budget. If nothing fits, say so.
- Answer in at most 10 lines: the city and why, the dates, the travel, the money left per person, the weather and what to pack."""

planner = create_agent(model=model, tools=TOOLS, system_prompt=RULES)
result = planner.invoke({"messages": [("user", USER_MESSAGE_GOAL)]})
show_run(result)

model asks for : next_weekend({})
model asks for : search_trip_guide({'query': 'Berlin to Prague bus fare travel time hostel prices free things to do'})
model asks for : search_trip_guide({'query': 'Berlin to Vienna bus fare travel time hostel prices free things to do'})
tool returns   : {"today": "2026-11-03", "saturday": "2026-11-07", "sunday": "2026-11-08"}
tool returns   : Prague, Czechia. Currency: Czech koruna (CZK). Bus from Berlin: about 4.5 hours, return fare about 40 EUR per person. Hostel bed: about 25 EUR per nig
tool returns   : Vienna, Austria. Currency: euro (EUR). Bus from Berlin: about 9.5 hours, return fare about 70 EUR per person. Hostel bed: about 35 EUR per night. Chea
model asks for : budget_per_person({'total_budget_eur': 300, 'people': 2, 'nights': 1, 'return_fare_eur': 40, 'bed_per_night_eur': 25})
model asks for : budget_per_person({'total_budget_eur': 300, 'people': 2, 'nights': 1, 'return_fare_eur': 70, 'bed_per_night_eur': 35})
model asks for : weather_fore

### What the better answer cost

Every model reply carries `usage_metadata`: the tokens the model read (`input_tokens`) and wrote (`output_tokens`). The helper adds them up for one run.

In [16]:
def count_usage(result: dict) -> dict:
    """Count model calls, tool calls and tokens of one agent run."""
    replies = [message for message in result["messages"] if message.type == "ai"]
    return {
        "model_calls": len(replies),
        "tool_calls": sum(len(reply.tool_calls) for reply in replies),
        "input_tokens": sum(reply.usage_metadata["input_tokens"] for reply in replies),
        "output_tokens": sum(reply.usage_metadata["output_tokens"] for reply in replies),
    }


print(count_usage(result))

{'model_calls': 3, 'tool_calls': 7, 'input_tokens': 6171, 'output_tokens': 1427}


Each model call sends the whole conversation again, with all tool descriptions. So the input grows with every step, and an agent with many steps reads its first question many times. At the Haiku 5.5 price on the Anthropic pricing page (3 November 2026: 0.10 USD per million input tokens, 0.50 USD per million output tokens), a run like this costs a fraction of a cent. Compare Step 1: one model call, no tools, and an answer full of guesses.

A system prompt is a request, not a guarantee: the model may still break a rule. A rule that must always hold goes into code, for example into a tool.

## Step 6: memory for a follow-up question

**Predict.** We ask a follow-up: "A third friend joins, so we are 3 people with 400 EUR. Does the plan still work?" What does `planner` answer?

In [17]:
USER_MESSAGE_FOLLOW_UP = "A third friend joins, so we are 3 people with 400 EUR. Does the plan still work?"
print(planner.invoke({"messages": [("user", USER_MESSAGE_FOLLOW_UP)]})["messages"][-1].text)

I don't have your earlier plan in this conversation, so I can't check it yet. To recalculate for 3 people with 400 EUR, I need:

1. **Destination city** (or I can search the student guide for options).
2. **Dates or number of nights**, so I can check the weather and the bed costs.
3. **Return fare per person and bed price per night**, if you already have them. If not, I'll look them up in the travel guide. Those guide prices are course examples, not live prices.

Once I have these, I'll run the budget for 3 people and tell you what is left per person.


The planner does not know which plan we mean: every `invoke` starts an empty conversation. **Short-term memory** keeps the messages of one conversation between calls.

- A **checkpointer** saves the conversation after every step. `InMemorySaver` from `langgraph.checkpoint.memory` keeps it in the notebook's memory.
- A **thread** is one conversation, named by a `thread_id` in the config: `{"configurable": {"thread_id": "trip-1"}}`. The same `thread_id` continues it; a new one starts fresh.

```mermaid
graph LR;
    call_1[invoke 1: USER_MESSAGE_GOAL, thread trip-1] --> agent_node[memory_planner];
    agent_node -- saves the messages --> saver[(InMemorySaver: thread trip-1)];
    call_2[invoke 2: USER_MESSAGE_FOLLOW_UP, thread trip-1] --> agent_node;
    saver -- loads the earlier messages --> agent_node;
    agent_node --> answer[Answer to the follow-up];
```

In [18]:
from langgraph.checkpoint.memory import InMemorySaver

memory_planner = create_agent(model=model, tools=TOOLS, system_prompt=RULES, checkpointer=InMemorySaver())
thread = {"configurable": {"thread_id": "trip-1"}}

memory_planner.invoke({"messages": [("user", USER_MESSAGE_GOAL)]}, thread)
follow_up_result = memory_planner.invoke({"messages": [("user", USER_MESSAGE_FOLLOW_UP)]}, thread)
print(follow_up_result["messages"][-1].text)

Yes, the plan still works with 3 people and 400 EUR, and Prague is still the better choice.

- **Prague:** 68.33 EUR per person left for food and activities (down from 85 EUR).
- **Vienna:** 28.33 EUR per person left (down from 45 EUR), so it fits but is tight.
- **Dates, weather and packing:** unchanged from before (Saturday 7 to Sunday 8 November, cold and mostly dry, so pack warm layers).
- **Assumption:** I counted one bed per person for one night. The 40 EUR and 70 EUR return fares and the 25 EUR and 35 EUR beds come from the travel guide, so they are course examples and should be checked before booking.


Now the agent remembers the plan and usually calls only `budget_per_person` again: the dates, the weather and the prices are already in the conversation. Memory saves tool calls, but every new call reads the whole saved conversation as input.

This is the end of Part 1: one agent with four tools, a knowledge base, rules and memory. Compare its answer with the one from Step 1.

## Step 7 (Part 2): a team of agents

### Why a team

Our planner researches Prague and Vienna one after the other, in one conversation that grows with every result. The two cities do not depend on each other, so two agents could research them **at the same time**.

**Orchestrator-workers** is a pattern for this: an orchestrator splits the task into subtasks, one **worker** solves each subtask, and a **synthesiser** combines the results. We build it with LangGraph, the library that `create_agent` runs on.

```mermaid
graph LR;
    request[Request] --> orchestrator[Orchestrator: one task per city];
    orchestrator --> prague[Worker: Prague];
    orchestrator --> vienna[Worker: Vienna];
    prague --> synthesiser[Synthesiser: compare and recommend];
    vienna --> synthesiser;
```

### The orchestrator: a plan as structured output

**Structured output** means the model's reply must fit a schema. The schema is a Pydantic class: each field has a type and a description, which the model reads like a docstring. `model.with_structured_output(TripPlan, method="json_schema")` returns a `TripPlan` object instead of text.

In [19]:
from pydantic import BaseModel, Field


class TripPlan(BaseModel):
    """A trip request split into one task per city."""

    people: int = Field(description="Number of travellers")
    budget_eur: float = Field(description="Total budget of the whole group in euros")
    nights: int = Field(description="Number of nights; 1 for a weekend")
    cities: list[str] = Field(description="The destination cities the user mentions, in English")


plan_model = model.with_structured_output(TripPlan, method="json_schema")
print(plan_model.invoke(USER_MESSAGE_GOAL))

people=2 budget_eur=300.0 nights=1 cities=['Prague', 'Vienna']


### The workers and the synthesiser

A graph shares a **state**, a dictionary that every node reads and updates. Two workers write their reports at the same time, so the key `reports` needs a rule for combining them: `Annotated[list[str], operator.add]` says "add the lists together". This rule is called a **reducer**.

- `orchestrator` makes the plan.
- `fan_out` returns one `Send("city_worker", {...})` per city. `Send` from `langgraph.types` starts one worker with its own input; all workers run in the same step, at the same time.
- `city_worker` runs a `create_agent` agent with our four tools on one city only, and returns its answer as a report.
- `synthesiser` is one model call: it reads the reports and recommends one city.

The whole graph, with what each node reads and writes. Each worker is a full agent with its own tool loop, and the reducer joins the two reports into one list.

```mermaid
graph TD;
    start_node([START: request = USER_MESSAGE_GOAL]) --> orchestrator_node[orchestrator: one model call with structured output];
    orchestrator_node -- writes plan: TripPlan --> fan_out_node{fan_out: one Send per city};
    fan_out_node -- "Send city = Prague" --> worker_prague;
    fan_out_node -- "Send city = Vienna" --> worker_vienna;
    subgraph worker_prague [city_worker: Prague]
        model_p[Model call] --> tools_p[Tools] --> model_p;
    end
    subgraph worker_vienna [city_worker: Vienna]
        model_v[Model call] --> tools_v[Tools] --> model_v;
    end
    worker_prague -- "reports: [Prague report]" --> reducer_node[reducer operator.add: one list of reports];
    worker_vienna -- "reports: [Vienna report]" --> reducer_node;
    reducer_node --> synthesiser_node[synthesiser: one model call, no tools];
    synthesiser_node -- writes recommendation --> end_node([END]);
```

In [20]:
import operator
from typing import Annotated

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send
from typing_extensions import TypedDict


class TripState(TypedDict):
    request: str
    plan: TripPlan
    reports: Annotated[list[str], operator.add]
    recommendation: str


worker_agent = create_agent(model=model, tools=TOOLS, system_prompt=RULES + "\nResearch only the one city you are given.")


def orchestrator(state: TripState) -> dict:
    """Split the request into one task per city."""
    return {"plan": plan_model.invoke(state["request"])}


def fan_out(state: TripState) -> list[Send]:
    """Start one worker per city in the plan."""
    return [Send("city_worker", {"city": city, "plan": state["plan"]}) for city in state["plan"].cities]


def city_worker(job: dict) -> dict:
    """Research one city with the worker agent and return its report."""
    plan = job["plan"]
    brief = (f"Research {job['city']} for this weekend: from Berlin, {plan.people} people, "
             f"{plan.budget_eur} EUR in total, {plan.nights} night(s).")
    result = worker_agent.invoke({"messages": [("user", brief)]})
    return {"reports": [f"## {job['city']}\n{result['messages'][-1].text}"]}


def synthesiser(state: TripState) -> dict:
    """Compare the reports and recommend one city."""
    reports = "\n\n".join(state["reports"])
    reply = model.invoke([("system", "You compare trip options for students and recommend one city."),
                          ("user", f"Request: {state['request']}\n\nReports:\n{reports}\n\nRecommend one city in at most 6 lines.")])
    return {"recommendation": reply.text}

### Wiring the graph

`StateGraph(TripState)` starts a graph; `add_node` adds a function as a node; `add_edge(a, b)` runs `b` after `a`. `add_conditional_edges("orchestrator", fan_out, ["city_worker"])` lets `fan_out` decide what runs next; the list names the nodes it can start. `compile()` makes the graph runnable.

In [21]:
builder = StateGraph(TripState)
builder.add_node("orchestrator", orchestrator)
builder.add_node("city_worker", city_worker)
builder.add_node("synthesiser", synthesiser)
builder.add_edge(START, "orchestrator")
builder.add_conditional_edges("orchestrator", fan_out, ["city_worker"])
builder.add_edge("city_worker", "synthesiser")
builder.add_edge("synthesiser", END)
trip_graph = builder.compile()

**Predict.** Which finishes first: the single `planner` or the team? `time.perf_counter()` is a clock in seconds, so the difference of two readings is a duration.

In [22]:
import time

start = time.perf_counter()
team_result = trip_graph.invoke({"request": USER_MESSAGE_GOAL})
team_seconds = time.perf_counter() - start

start = time.perf_counter()
planner.invoke({"messages": [("user", USER_MESSAGE_GOAL)]})
single_seconds = time.perf_counter() - start

for report in team_result["reports"]:
    print(report, "\n")
print("RECOMMENDATION:\n" + team_result["recommendation"])
print(f"\nteam: {team_seconds:.0f} s, single agent: {single_seconds:.0f} s")

## Prague
- **City and why:** Prague, Czechia. It's the closest option in the guide, with cheap food and free sights like Charles Bridge at dawn and the view from Letna Park.
- **Dates:** Saturday 7 to Sunday 8 November 2026, 1 night.
- **Travel:** Bus from Berlin, about 4.5 hours each way. Return fare is about 40 EUR per person.
- **Bed:** Hostel, about 25 EUR per night.
- **Money left:** 85 EUR per person for food and activities (150 EUR per person after the fare and bed: 300 EUR total ÷ 2 people, minus 40 EUR fare and 25 EUR bed). The plan is within budget.
- **Price note:** Fares, bed and food prices come from the student travel guide and are course examples, not live prices.
- **Weather Saturday:** 1.5 to 6.6 °C, 6% chance of rain.
- **Weather Sunday:** -0.8 to 4.6 °C, 9% chance of rain.
- **What to pack:** Warm waterproof coat, hat, gloves and a scarf, plus layers for the cold mornings.
- **Currency:** The guide lists the Czech koruna (CZK), so bring some cash or check card fees.

### When is a team worth it?

The workers ran at the same time, so the team waits about as long as the slowest worker, plus the orchestrator and the synthesiser. But the team makes more model calls: one for the plan, one run per worker, one to combine. Network and model times vary, so run the cell twice before you conclude.

A simple guideline: stay with one agent while one conversation can hold the task. Move to a team when the task splits into **independent** parts (one per city, per document, per product), when there are several of them, and when the waiting time matters. With one city, a team only adds calls.

## Check your understanding

Vote alone first, then discuss with your neighbour, then vote again.

**1.** In Step 1, the agent described the weather for the weekend. Where did that come from?

- A. From a weather service that the model calls by itself.
- B. From the model's training data and guessing: it had no tool and does not know today's date.
- C. From the system prompt.
- D. From LangChain, which adds the weather to every request.

**2.** Who runs the Python function of a tool?

- A. Claude, on Anthropic's servers.
- B. Our own Python process: the model only asks for the call.
- C. The web service of the tool.
- D. Nobody; the model imagines the result.

**3.** Why does the budget come from the tool `budget_per_person` and not from the model?

- A. The model is not allowed to see numbers.
- B. A sum in Python is exact and the same every time; a model can make arithmetic mistakes.
- C. Tools are cheaper than tokens.
- D. The system prompt cannot mention money.

**4.** `planner` does not understand the follow-up question, but `memory_planner` does. What is the difference?

- A. `memory_planner` uses a larger model.
- B. `memory_planner` has a checkpointer, and both calls use the same `thread_id`, so the first conversation is saved and read again.
- C. `memory_planner` has more tools.
- D. The follow-up was asked twice.

**5.** The team of agents made more model calls than the single planner but can finish sooner. Why?

- A. Haiku answers faster inside a graph.
- B. The workers ran at the same time, so the waiting time follows the slowest worker, not the sum.
- C. `Send` saves results so later calls are skipped.
- D. The synthesiser has no tools.

## Exercises

Work in pairs. Every exercise needs your API key.

### 5.1 · Predict the trajectory

*Predict*

> Without running anything, write down which tools `planner` will call for the question *"What should I pack for Prague this weekend?"*, in which order, and which tools it will not need. Then run it with `show_run` and compare. Where did the model surprise you?

*Source:* course-written for this workbook.

*Your answer:*

### 5.2 · Add a city to the knowledge base

*Code*

> Add a note for Leipzig to `TRIP_GUIDE`, in the same form as the others (your own example prices). Rebuild `planner` and ask: *"Leipzig or Dresden this weekend, 2 people, 150 EUR?"* Check in the `tool returns` lines that the agent found your note. Then remove the hostel price from your note and ask again: does the agent say that the price is missing, or does it guess?

*Source:* course-written for this workbook.

### 5.3 · A currency tool with a second web service

*Code*

> Prague uses the Czech koruna. Write a tool `convert_currency(amount: float, from_currency: str, to_currency: str) -> str` with the free Frankfurter service: `httpx.get("https://api.frankfurter.dev/v2/rates", params={"base": "EUR", "quotes": "CZK"})` returns a list such as `[{"date": "...", "base": "EUR", "quote": "CZK", "rate": ...}]`. Return the converted amount, the rate and its date, and an "Error: ..." sentence if the request fails. Test it by hand, add it to the tools, and ask: *"How much is 85 EUR in Prague money?"*

*Source:* course-written for this workbook; API: [Frankfurter](https://frankfurter.dev/).

### 5.4 · A question the tools cannot answer

*Predict*

> Ask `planner` about a weekend more than 16 days from today, for example *"We want to go to Vienna in four weeks. What should we pack?"* The weather tool will return an error. Predict first: will the agent stop, try again, or answer without a forecast? Then run it. Is the answer honest about what it does not know?

*Source:* course-written for this workbook.

### 5.5 · A bigger team

*Pairs*

> Change `USER_MESSAGE_GOAL` to three cities: Prague, Vienna or Dresden. Run the team and the single planner again and note both times. Which gains more from the extra city? Write two sentences of advice for a team that must choose between one agent and a team of agents.

*Source:* course-written for this workbook.

## Debrief

- **Compare the answers of Step 1 and Step 5** for the same question: list each number in the Step 5 answer and the tool it came from.
- **Compare two pairs' trajectories** for exercise 5.1. They are usually not the same: an agent is not deterministic.
- **Exercise 5.2:** which pairs saw the agent guess a missing price? A knowledge base helps only if the agent says when it finds nothing.
- **Exercise 5.5:** a choice of architecture should quote measured times, not a general preference.

## Further reading

- LangChain docs: [Agents](https://docs.langchain.com/oss/python/langchain/agents), [Tools](https://docs.langchain.com/oss/python/langchain/tools), [Short-term memory](https://docs.langchain.com/oss/python/langchain/short-term-memory), [Structured output](https://docs.langchain.com/oss/python/langchain/structured-output).
- LangGraph docs: [Workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents), section *Orchestrator-worker*, the source of the `Send` pattern in Step 7.
- Anthropic: [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents), on workflows, orchestrator-workers and when an agent is worth it; [Pricing](https://platform.claude.com/docs/en/about-claude/pricing), read on 3 November 2026.
- Notebooks 01, 03 and 04 of this session: retrieval with embeddings, the LangChain syntax in full, and more multi-agent patterns.
- Open-Meteo: [forecast docs](https://open-meteo.com/en/docs), [geocoding docs](https://open-meteo.com/en/docs/geocoding-api), [terms](https://open-meteo.com/en/terms) (free for non-commercial use, data under CC BY 4.0).
- Frankfurter (exercise 5.3): [docs](https://frankfurter.dev/), [licence and data providers](https://frankfurter.dev/license/).